# Feature Engineering

This notebook constructs the machine learning dataset used by the Demand Prediction Agent.

Temporal, spatial, pricing, and infrastructure features are combined into a unified modeling table for occupancy forecasting.

In [3]:
import pandas as pd
import numpy as np

occupancy = pd.read_csv("../data/urbanev/occupancy.csv")
time = pd.read_csv("../data/urbanev/time.csv")
information = pd.read_csv("../data/urbanev/information.csv")
adj = pd.read_csv("../data/urbanev/adj.csv")
price = pd.read_csv("../data/urbanev/price.csv")

time["timestamp"] = pd.to_datetime(
    time[["year","month","day","hour","minute","second"]]
)

time.head()

,month,day,year,hour,minute,second,timestamp
0,6,19,2022,0,0,0,2022-06-19 00:00:00
1,6,19,2022,0,5,0,2022-06-19 00:05:00
2,6,19,2022,0,10,0,2022-06-19 00:10:00
3,6,19,2022,0,15,0,2022-06-19 00:15:00
4,6,19,2022,0,20,0,2022-06-19 00:20:00


In [4]:
occ_long = occupancy.melt(
    id_vars=["timestamp"],
    var_name="grid",
    value_name="occupancy"
)

occ_long["grid"] = occ_long["grid"].astype(str)

In [5]:
price_long = price.melt(
    id_vars=["timestamp"],
    var_name="grid",
    value_name="price"
)

price_long["grid"] = price_long["grid"].astype(str)

In [6]:
timestamp_map = pd.DataFrame({
    "timestamp_id": occupancy["timestamp"],
    "datetime": time["timestamp"]
})
occ_long = occ_long.rename(
    columns={"timestamp":"timestamp_id"}
)

price_long = price_long.rename(
    columns={"timestamp":"timestamp_id"}
)

In [7]:
occ_long = occ_long.merge(
    timestamp_map,
    on="timestamp_id",
    how="left"
)

price_long = price_long.merge(
    timestamp_map,
    on="timestamp_id",
    how="left"
)

In [8]:
df = occ_long.merge(
    price_long[
        ["timestamp_id","grid","price"]
    ],
    on=["timestamp_id","grid"]
)

In [9]:
print(occ_long.shape)

print(price_long.shape)

print(df.shape)

(2134080, 4)
(2134080, 4)
(2134080, 5)


In [10]:
info = information[
    [
        "grid",
        "count",
        "fast_count",
        "slow_count",
        "area",
        "CBD",
        "dynamic_pricing"
    ]
].copy()

info["grid"] = info["grid"].astype(str)

df = df.merge(
    info,
    on="grid",
    how="left"
)

print(df.shape)

df.head()

(2134080, 11)


,timestamp_id,grid,occupancy,datetime,price,count,fast_count,slow_count,area,CBD,dynamic_pricing
0,1,102,12,2022-06-19 00:00:00,0.924,30,3,27,0.71,0,0
1,2,102,12,2022-06-19 00:05:00,0.924,30,3,27,0.71,0,0
2,3,102,12,2022-06-19 00:10:00,0.924,30,3,27,0.71,0,0
3,4,102,12,2022-06-19 00:15:00,0.924,30,3,27,0.71,0,0
4,5,102,12,2022-06-19 00:20:00,0.924,30,3,27,0.71,0,0


In [11]:
df["hour"] = df["datetime"].dt.hour

df["dayofweek"] = df["datetime"].dt.dayofweek

df["month"] = df["datetime"].dt.month

df["is_weekend"] = (
    df["dayofweek"] >= 5
).astype(int)

In [12]:
df = df.sort_values(
    ["grid", "datetime"]
).reset_index(drop=True)

In [13]:
df["lag_1"] = (
    df.groupby("grid")["occupancy"]
      .shift(1)
)

df["lag_12"] = (
    df.groupby("grid")["occupancy"]
      .shift(12)
)

df["lag_24"] = (
    df.groupby("grid")["occupancy"]
      .shift(24)
)

df["lag_288"] = (
    df.groupby("grid")["occupancy"]
      .shift(288)
)

In [14]:
df["rolling_mean_12"] = (
    df.groupby("grid")["occupancy"]
      .transform(
          lambda x:
          x.shift(1).rolling(12).mean()
      )
)

df["rolling_std_12"] = (
    df.groupby("grid")["occupancy"]
      .transform(
          lambda x:
          x.shift(1).rolling(12).std()
      )
)

In [15]:
df["target"] = (
    df.groupby("grid")["occupancy"]
      .shift(-12)
)

In [16]:
df = df.dropna()

In [17]:
df.shape

(2059980, 22)

In [18]:
df[
    [
        "occupancy",
        "lag_1",
        "lag_12",
        "lag_24",
        "lag_288",
        "target"
    ]
].head()

,occupancy,lag_1,lag_12,lag_24,lag_288,target
288,61,62.0,64.0,62.0,60.0,65.0
289,62,61.0,64.0,62.0,60.0,65.0
290,62,62.0,64.0,62.0,60.0,65.0
291,63,62.0,63.0,62.0,60.0,64.0
292,64,63.0,63.0,62.0,60.0,64.0


In [19]:
df = df.rename(
    columns={
        "count_x":"count",
        "fast_count_x":"fast_count",
        "slow_count_x":"slow_count",
        "area_x":"area",
        "CBD_x":"CBD",
        "dynamic_pricing_x":"dynamic_pricing"
    }
)

In [20]:
adj_matrix = adj.copy()

adj_matrix = adj_matrix.set_index(
    "node_id"
)

adj_matrix.head()

,102,105,107,108,109,110,111,115,123,124,...,1160,1162,1163,1164,1166,1167,1168,1170,1172,1173
node_id,,,,,,,,,,,,,,,,,,,,,
102,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
105,0,1,1,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
107,0,1,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
108,0,1,1,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
109,0,0,0,1,1,1,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0


In [21]:
neighbor_dict = {}

for grid in adj_matrix.index:

    neighbors = (
        adj_matrix.loc[grid]
        [adj_matrix.loc[grid] == 1]
        .index
        .tolist()
    )

    neighbors = [
        str(n)
        for n in neighbors
        if str(n) != str(grid)
    ]

    neighbor_dict[str(grid)] = neighbors

In [22]:
neighbor_dict["1167"]

['123', '124', '773', '974', '1138', '1164', '1166']

In [23]:
occ_wide = occupancy.copy()

occ_wide = occ_wide.drop(
    columns=["timestamp"]
)

In [24]:
neighbor_occ = pd.DataFrame()

for grid in occ_wide.columns:

    neighbors = neighbor_dict.get(
        str(grid),
        []
    )

    if len(neighbors) == 0:

        neighbor_occ[grid] = np.nan

    else:

        valid_neighbors = [
            n for n in neighbors
            if n in occ_wide.columns
        ]

        neighbor_occ[grid] = (
            occ_wide[
                valid_neighbors
            ].mean(axis=1)
        )

C:\Users\HP\AppData\Local\Temp\ipykernel_30864\1615470600.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  neighbor_occ[grid] = (
C:\Users\HP\AppData\Local\Temp\ipykernel_30864\1615470600.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  neighbor_occ[grid] = (
C:\Users\HP\AppData\Local\Temp\ipykernel_30864\1615470600.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(a

In [25]:
neighbor_occ["timestamp"] = occupancy["timestamp"]

neighbor_long = neighbor_occ.melt(
    id_vars=["timestamp"],
    var_name="grid",
    value_name="neighbor_occupancy"
)

neighbor_long["grid"] = (
    neighbor_long["grid"]
    .astype(str)
)
neighbor_long = neighbor_long.rename(
    columns={"timestamp": "timestamp_id"}
)

neighbor_long["grid"] = (
    neighbor_long["grid"]
    .astype(str)
)

C:\Users\HP\AppData\Local\Temp\ipykernel_30864\3901960503.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  neighbor_occ["timestamp"] = occupancy["timestamp"]


In [26]:
df = df.merge(
    neighbor_long,
    on=[
        "timestamp_id",
        "grid"
    ],
    how="left"
)

In [27]:
print(df.shape)

df[
    [
        "occupancy",
        "neighbor_occupancy"
    ]
].head()

(2059980, 23)


,occupancy,neighbor_occupancy
0,61,31.8
1,62,32.2
2,62,32.8
3,63,33.6
4,64,34.0


In [28]:
df["neighbor_occupancy"] = (
    df["neighbor_occupancy"]
    .fillna(df["occupancy"])
)

In [29]:
df["neighbor_occupancy"].isna().sum()

np.int64(0)

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2059980 entries, 0 to 2059979
Data columns (total 23 columns):
 #   Column              Dtype         
---  ------              -----         
 0   timestamp_id        int64         
 1   grid                str           
 2   occupancy           int64         
 3   datetime            datetime64[us]
 4   price               float64       
 5   count               int64         
 6   fast_count          int64         
 7   slow_count          int64         
 8   area                float64       
 9   CBD                 int64         
 10  dynamic_pricing     int64         
 11  hour                int32         
 12  dayofweek           int32         
 13  month               int32         
 14  is_weekend          int64         
 15  lag_1               float64       
 16  lag_12              float64       
 17  lag_24              float64       
 18  lag_288             float64       
 19  rolling_mean_12     float64       
 20  rolling_std_1

In [31]:
df.memory_usage(deep=True).sum() / 1024**2

np.float64(344.6305179595947)

In [32]:
import os

os.makedirs(
    "../data/processed",
    exist_ok=True
)

In [33]:
df.to_parquet(
    "../data/processed/demand_features.parquet",
    index=False
)